In [2]:
!pip -q install pyspark

In [3]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark import StorageLevel
import time
spark = SparkSession.builder \
    .appName("PySparkPractice") \
    .getOrCreate()
products_data = [
    ("P01", "Laptop", "Electronics", 60000),
    ("P02", "Monitor", "Electronics", 14000),
    ("P03", "Headphones", "Accessories", 3500),
    ("P04", "Printer", "Office", 17000),
    ("P05", "Router", "Networking", 4500),
    ("P06", "Tablet", "Electronics", 24000)
]
products_df = spark.createDataFrame(
    products_data,
    ["product_id", "product_name", "category", "list_price"]
)
stores_data = [
    ("S01", "Hyderabad", "South"),
    ("S02", "Bengaluru", "South"),
    ("S03", "Mumbai", "West"),
    ("S04", "Delhi", "North")
]
stores_df = spark.createDataFrame(
    stores_data,
    ["store_id", "city", "region"]
)
products_df.show()
stores_df.show()
sales_df = (
    spark.range(1, 100001)
    .withColumnRenamed("id", "order_id")
    .withColumn(
        "order_date",
        F.date_add(
            F.to_date(F.lit("2026-01-01")),
            (F.col("order_id") % 180).cast("int")
        )
    )
    .withColumn(
        "product_id",
        F.concat(
            F.lit("P"),
            F.lpad(
                ((F.col("order_id") % 6) + 1).cast("string"),
                2, "0"
            )
        )
    )
    .withColumn(
        "store_id",
        F.concat(
            F.lit("S"),
            F.lpad(
                ((F.floor(F.col("order_id") / 3) % 4) + 1)
                .cast("string"),
                2, "0"
            )
        )
    )
    .withColumn(
        "quantity",
        ((F.col("order_id") % 8) + 1).cast("int")
    )
    .withColumn(
        "discount_pct",
        F.when(
            F.col("order_id") % 19 == 0,
            F.lit(None).cast("int")
        ).otherwise(
            ((F.floor(F.col("order_id") / 5) % 4) * 5)
            .cast("int")
        )
    )
    .withColumn(
        "order_status",
        F.when(F.col("order_id") % 11 == 0, "Cancelled")
         .when(F.col("order_id") % 7 == 0, "Returned")
         .otherwise("Completed")
    )

    .withColumn(
        "payment_method",
        F.when(F.col("order_id") % 3 == 0, "UPI")
         .when(F.col("order_id") % 3 == 1, "Card")
         .otherwise("Cash")
    )

    .withColumn(
        "salesperson_id",
        F.when(
            F.col("order_id") % 17 == 0,
            F.lit(None).cast("string")
        ).otherwise(
            F.concat(
                F.lit("EMP"),
                F.lpad(
                    ((F.col("order_id") % 12) + 1).cast("string"),
                    2, "0"
                )
            )
        )
    )

    .join(
        products_df.select(
            "product_id",
            F.col("list_price").alias("unit_price")
        ),
        "product_id",
        "left"
    )

    .select(
        "order_id", "order_date", "store_id",
        "product_id", "quantity", "unit_price",
        "discount_pct", "order_status",
        "payment_method", "salesperson_id"
    )
)

print("Total Records:", sales_df.count())
sales_df.printSchema()
sales_df.show(10, truncate=False)


+----------+------------+-----------+----------+
|product_id|product_name|   category|list_price|
+----------+------------+-----------+----------+
|       P01|      Laptop|Electronics|     60000|
|       P02|     Monitor|Electronics|     14000|
|       P03|  Headphones|Accessories|      3500|
|       P04|     Printer|     Office|     17000|
|       P05|      Router| Networking|      4500|
|       P06|      Tablet|Electronics|     24000|
+----------+------------+-----------+----------+

+--------+---------+------+
|store_id|     city|region|
+--------+---------+------+
|     S01|Hyderabad| South|
|     S02|Bengaluru| South|
|     S03|   Mumbai|  West|
|     S04|    Delhi| North|
+--------+---------+------+

Total Records: 100000
root
 |-- order_id: long (nullable = false)
 |-- order_date: date (nullable = true)
 |-- store_id: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- unit_price: long (nullable = true)
 |-- discount_p

**Part A**

In [4]:
sales_df.select(
    "order_id",
    "product_id",
    "quantity",
    "unit_price"
).show()

+--------+----------+--------+----------+
|order_id|product_id|quantity|unit_price|
+--------+----------+--------+----------+
|       3|       P04|       4|     17000|
|       9|       P04|       2|     17000|
|      15|       P04|       8|     17000|
|      21|       P04|       6|     17000|
|       2|       P03|       3|      3500|
|       8|       P03|       1|      3500|
|      14|       P03|       7|      3500|
|      20|       P03|       5|      3500|
|       4|       P05|       5|      4500|
|      10|       P05|       3|      4500|
|      16|       P05|       1|      4500|
|       5|       P06|       6|     24000|
|      11|       P06|       4|     24000|
|      17|       P06|       2|     24000|
|       1|       P02|       2|     14000|
|       7|       P02|       8|     14000|
|      13|       P02|       6|     14000|
|      19|       P02|       4|     14000|
|       6|       P01|       7|     60000|
|      12|       P01|       5|     60000|
+--------+----------+--------+----

In [5]:
sales_df.filter(
    (F.col("order_status") == "Completed") &
    (F.col("quantity") > 5)
).show()

+--------+----------+--------+----------+--------+----------+------------+------------+--------------+--------------+
|order_id|order_date|store_id|product_id|quantity|unit_price|discount_pct|order_status|payment_method|salesperson_id|
+--------+----------+--------+----------+--------+----------+------------+------------+--------------+--------------+
|      15|2026-01-16|     S02|       P04|       8|     17000|          15|   Completed|           UPI|         EMP04|
|      39|2026-02-09|     S02|       P04|       8|     17000|          15|   Completed|           UPI|         EMP04|
|      45|2026-02-15|     S04|       P04|       6|     17000|           5|   Completed|           UPI|         EMP10|
|      69|2026-03-11|     S04|       P04|       6|     17000|           5|   Completed|           UPI|         EMP10|
|      38|2026-02-08|     S01|       P03|       7|      3500|        NULL|   Completed|          Cash|         EMP03|
|      62|2026-03-04|     S01|       P03|       7|      

In [6]:
sales_revenue_df = sales_df.withColumn(
    "gross_revenue",
    F.col("quantity") * F.col("unit_price")
)

sales_revenue_df.show()

+--------+----------+--------+----------+--------+----------+------------+------------+--------------+--------------+-------------+
|order_id|order_date|store_id|product_id|quantity|unit_price|discount_pct|order_status|payment_method|salesperson_id|gross_revenue|
+--------+----------+--------+----------+--------+----------+------------+------------+--------------+--------------+-------------+
|       3|2026-01-04|     S02|       P04|       4|     17000|           0|   Completed|           UPI|         EMP04|        68000|
|       9|2026-01-10|     S04|       P04|       2|     17000|           5|   Completed|           UPI|         EMP10|        34000|
|      15|2026-01-16|     S02|       P04|       8|     17000|          15|   Completed|           UPI|         EMP04|       136000|
|      21|2026-01-22|     S04|       P04|       6|     17000|           0|    Returned|           UPI|         EMP10|       102000|
|       2|2026-01-03|     S01|       P03|       3|      3500|           0|  

In [7]:
renamed_df = sales_df.withColumnRenamed(
    "order_status",
    "transaction_status"
).drop("payment_method")

renamed_df.show()

+--------+----------+--------+----------+--------+----------+------------+------------------+--------------+
|order_id|order_date|store_id|product_id|quantity|unit_price|discount_pct|transaction_status|salesperson_id|
+--------+----------+--------+----------+--------+----------+------------+------------------+--------------+
|       3|2026-01-04|     S02|       P04|       4|     17000|           0|         Completed|         EMP04|
|       9|2026-01-10|     S04|       P04|       2|     17000|           5|         Completed|         EMP10|
|      15|2026-01-16|     S02|       P04|       8|     17000|          15|         Completed|         EMP04|
|      21|2026-01-22|     S04|       P04|       6|     17000|           0|          Returned|         EMP10|
|       2|2026-01-03|     S01|       P03|       3|      3500|           0|         Completed|         EMP03|
|       8|2026-01-09|     S03|       P03|       1|      3500|           5|         Completed|         EMP09|
|      14|2026-01-1

**Part B**

In [8]:
sales_df.groupBy("product_id").agg(
    F.sum("quantity").alias("total_quantity")
).show()

+----------+--------------+
|product_id|total_quantity|
+----------+--------------+
|       P04|         83334|
|       P03|         66667|
|       P05|         66665|
|       P06|         83330|
|       P02|         83336|
|       P01|         66668|
+----------+--------------+



In [9]:
sales_df.join(
    products_df,
    "product_id"
).withColumn(
    "gross_revenue",
    F.col("quantity") * F.col("unit_price")
).groupBy("category").agg(
    F.sum("gross_revenue").alias("total_gross_revenue")
).show()

+-----------+-------------------+
|   category|total_gross_revenue|
+-----------+-------------------+
|     Office|         1416678000|
| Networking|          299992500|
|Electronics|         7166704000|
|Accessories|          233334500|
+-----------+-------------------+



In [10]:
sales_df.join(
    stores_df,
    "store_id"
).groupBy("city").agg(
    F.count("*").alias("total_transactions")
).show()

+---------+------------------+
|     city|total_transactions|
+---------+------------------+
|   Mumbai|             24999|
|    Delhi|             24999|
|Bengaluru|             25001|
|Hyderabad|             25001|
+---------+------------------+



In [11]:
top_products_df = sales_df.filter(
    F.col("order_status") == "Completed"
).join(
    products_df,
    "product_id"
).join(
    stores_df,
    "store_id"
).withColumn(
    "gross_revenue",
    F.col("quantity") * F.col("unit_price")
).groupBy(
    "product_id", "product_name"
).agg(
    F.sum("gross_revenue").alias("total_gross_revenue")
).orderBy(
    F.col("total_gross_revenue").desc()
)

top_products_df.show(5)

+----------+------------+-------------------+
|product_id|product_name|total_gross_revenue|
+----------+------------+-------------------+
|       P01|      Laptop|         3116940000|
|       P06|      Tablet|         1558416000|
|       P04|     Printer|         1103912000|
|       P02|     Monitor|          909076000|
|       P05|      Router|          233766000|
+----------+------------+-------------------+
only showing top 5 rows


**Part C**

In [12]:
sales_df.select(
    F.sum(
        F.when(F.col("discount_pct").isNull(), 1).otherwise(0)
    ).alias("null_discount_count"),
    F.sum(
        F.when(F.col("salesperson_id").isNull(), 1).otherwise(0)
    ).alias("null_salesperson_count")
).show()

+-------------------+----------------------+
|null_discount_count|null_salesperson_count|
+-------------------+----------------------+
|               5263|                  5882|
+-------------------+----------------------+



In [13]:
filled_df = sales_df.na.fill({
    "discount_pct": 0
})

filled_df.show()

+--------+----------+--------+----------+--------+----------+------------+------------+--------------+--------------+
|order_id|order_date|store_id|product_id|quantity|unit_price|discount_pct|order_status|payment_method|salesperson_id|
+--------+----------+--------+----------+--------+----------+------------+------------+--------------+--------------+
|       3|2026-01-04|     S02|       P04|       4|     17000|           0|   Completed|           UPI|         EMP04|
|       9|2026-01-10|     S04|       P04|       2|     17000|           5|   Completed|           UPI|         EMP10|
|      15|2026-01-16|     S02|       P04|       8|     17000|          15|   Completed|           UPI|         EMP04|
|      21|2026-01-22|     S04|       P04|       6|     17000|           0|    Returned|           UPI|         EMP10|
|       2|2026-01-03|     S01|       P03|       3|      3500|           0|   Completed|          Cash|         EMP03|
|       8|2026-01-09|     S03|       P03|       1|      

In [14]:
clean_sales_df = sales_df.na.drop(
    subset=["salesperson_id"]
)

clean_sales_df.show()

+--------+----------+--------+----------+--------+----------+------------+------------+--------------+--------------+
|order_id|order_date|store_id|product_id|quantity|unit_price|discount_pct|order_status|payment_method|salesperson_id|
+--------+----------+--------+----------+--------+----------+------------+------------+--------------+--------------+
|       3|2026-01-04|     S02|       P04|       4|     17000|           0|   Completed|           UPI|         EMP04|
|       9|2026-01-10|     S04|       P04|       2|     17000|           5|   Completed|           UPI|         EMP10|
|      15|2026-01-16|     S02|       P04|       8|     17000|          15|   Completed|           UPI|         EMP04|
|      21|2026-01-22|     S04|       P04|       6|     17000|           0|    Returned|           UPI|         EMP10|
|       2|2026-01-03|     S01|       P03|       3|      3500|           0|   Completed|          Cash|         EMP03|
|       8|2026-01-09|     S03|       P03|       1|      

In [15]:
order_size_df = sales_df.withColumn(
    "order_size",
    F.when(F.col("quantity") <= 3, "Small")
    .when(F.col("quantity") <= 6, "Medium")
    .otherwise("Large")
)

order_size_df.show()

+--------+----------+--------+----------+--------+----------+------------+------------+--------------+--------------+----------+
|order_id|order_date|store_id|product_id|quantity|unit_price|discount_pct|order_status|payment_method|salesperson_id|order_size|
+--------+----------+--------+----------+--------+----------+------------+------------+--------------+--------------+----------+
|       3|2026-01-04|     S02|       P04|       4|     17000|           0|   Completed|           UPI|         EMP04|    Medium|
|       9|2026-01-10|     S04|       P04|       2|     17000|           5|   Completed|           UPI|         EMP10|     Small|
|      15|2026-01-16|     S02|       P04|       8|     17000|          15|   Completed|           UPI|         EMP04|     Large|
|      21|2026-01-22|     S04|       P04|       6|     17000|           0|    Returned|           UPI|         EMP10|    Medium|
|       2|2026-01-03|     S01|       P03|       3|      3500|           0|   Completed|          

**Part D**

In [16]:
product_revenue_df = sales_df.join(
    products_df,
    "product_id"
).withColumn(
    "gross_revenue",
    F.col("quantity") * F.col("unit_price")
).groupBy(
    "category", "product_id", "product_name"
).agg(
    F.sum("gross_revenue").alias("total_revenue")
)

window_spec = Window.partitionBy(
    "category"
).orderBy(
    F.col("total_revenue").desc()
)

ranked_products_df = product_revenue_df.withColumn(
    "revenue_rank",
    F.dense_rank().over(window_spec)
)

ranked_products_df.show()

+-----------+----------+------------+-------------+------------+
|   category|product_id|product_name|total_revenue|revenue_rank|
+-----------+----------+------------+-------------+------------+
|Accessories|       P03|  Headphones|    233334500|           1|
|Electronics|       P01|      Laptop|   4000080000|           1|
|Electronics|       P06|      Tablet|   1999920000|           2|
|Electronics|       P02|     Monitor|   1166704000|           3|
| Networking|       P05|      Router|    299992500|           1|
|     Office|       P04|     Printer|   1416678000|           1|
+-----------+----------+------------+-------------+------------+



In [17]:
store_window = Window.partitionBy(
    "store_id"
).orderBy(
    "order_date", "order_id"
).rowsBetween(
    Window.unboundedPreceding,
    Window.currentRow
)

running_revenue_df = sales_df.withColumn(
    "gross_revenue",
    F.col("quantity") * F.col("unit_price")
).withColumn(
    "running_gross_revenue",
    F.sum("gross_revenue").over(store_window)
)

running_revenue_df.show()

+--------+----------+--------+----------+--------+----------+------------+------------+--------------+--------------+-------------+---------------------+
|order_id|order_date|store_id|product_id|quantity|unit_price|discount_pct|order_status|payment_method|salesperson_id|gross_revenue|running_gross_revenue|
+--------+----------+--------+----------+--------+----------+------------+------------+--------------+--------------+-------------+---------------------+
|     180|2026-01-01|     S01|       P01|       5|     60000|           0|   Completed|           UPI|         EMP01|       300000|               300000|
|     360|2026-01-01|     S01|       P01|       1|     60000|           0|   Completed|           UPI|         EMP01|        60000|               360000|
|     540|2026-01-01|     S01|       P01|       5|     60000|           0|   Completed|           UPI|         EMP01|       300000|               660000|
|     720|2026-01-01|     S01|       P01|       1|     60000|           0|  

In [18]:
monthly_sales_df = sales_df.filter(
    F.col("order_status") == "Completed"
).withColumn(
    "month",
    F.month("order_date")
).withColumn(
    "gross_revenue",
    F.col("quantity") * F.col("unit_price")
).groupBy(
    "month"
).agg(
    F.sum("gross_revenue").alias("monthly_revenue")
).orderBy("month")

monthly_sales_df.show()

+-----+---------------+
|month|monthly_revenue|
+-----+---------------+
|    1|     1278728500|
|    2|     1027585500|
|    3|     1212087500|
|    4|     1218921000|
|    5|     1277560500|
|    6|     1089027500|
+-----+---------------+



**Part E**

In [19]:
step1_df = sales_df.filter(
    F.col("quantity") > 5
)

print("Transformation 1 created")

step2_df = step1_df.withColumn(
    "gross_revenue",
    F.col("quantity") * F.col("unit_price")
)

print("Transformation 2 created")

step3_df = step2_df.select(
    "order_id",
    "product_id",
    "quantity",
    "gross_revenue"
)

print("Transformation 3 created")

step3_df.show()

Transformation 1 created
Transformation 2 created
Transformation 3 created
+--------+----------+--------+-------------+
|order_id|product_id|quantity|gross_revenue|
+--------+----------+--------+-------------+
|      15|       P04|       8|       136000|
|      21|       P04|       6|       102000|
|      39|       P04|       8|       136000|
|      45|       P04|       6|       102000|
|      14|       P03|       7|        24500|
|      38|       P03|       7|        24500|
|      22|       P05|       7|        31500|
|      46|       P05|       7|        31500|
|       5|       P06|       6|       144000|
|      23|       P06|       8|       192000|
|      29|       P06|       6|       144000|
|      47|       P06|       8|       192000|
|      53|       P06|       6|       144000|
|       7|       P02|       8|       112000|
|      13|       P02|       6|        84000|
|      31|       P02|       8|       112000|
|      37|       P02|       6|        84000|
|      55|       P02|    

In [20]:
aggregation_df = sales_df.groupBy(
    "product_id"
).agg(
    F.sum("quantity").alias("total_quantity"),
    F.sum(
        F.col("quantity") * F.col("unit_price")
    ).alias("total_revenue")
)

aggregation_df.explain()

aggregation_df.collect()

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- HashAggregate(keys=[product_id#33], functions=[sum(quantity#35), sum((cast(quantity#35 as bigint) * unit_price#40L))])
   +- HashAggregate(keys=[product_id#33], functions=[partial_sum(quantity#35), partial_sum((cast(quantity#35 as bigint) * unit_price#40L))])
      +- Project [product_id#33, quantity#35, unit_price#40L]
         +- SortMergeJoin [product_id#33], [product_id#0], LeftOuter
            :- Sort [product_id#33 ASC NULLS FIRST], false, 0
            :  +- Exchange hashpartitioning(product_id#33, 200), ENSURE_REQUIREMENTS, [plan_id=4424]
            :     +- Project [concat(P, lpad(cast(((id#30L % 6) + 1) as string), 2, 0)) AS product_id#33, cast(((id#30L % 8) + 1) as int) AS quantity#35]
            :        +- Range (1, 100001, step=1, splits=2)
            +- Sort [product_id#0 ASC NULLS FIRST], false, 0
               +- Exchange hashpartitioning(product_id#0, 200), ENSURE_REQUIREMENTS, [plan_id=4425]
            

[Row(product_id='P04', total_quantity=83334, total_revenue=1416678000),
 Row(product_id='P03', total_quantity=66667, total_revenue=233334500),
 Row(product_id='P05', total_quantity=66665, total_revenue=299992500),
 Row(product_id='P06', total_quantity=83330, total_revenue=1999920000),
 Row(product_id='P02', total_quantity=83336, total_revenue=1166704000),
 Row(product_id='P01', total_quantity=66668, total_revenue=4000080000)]

**Part F**

In [21]:
completed_df = sales_df.filter(
    F.col("order_status") == "Completed"
).cache()

start_time = time.time()
completed_df.count()
first_time = time.time() - start_time

start_time = time.time()
completed_df.count()
second_time = time.time() - start_time

print("First action time:", first_time, "seconds")
print("Second action time:", second_time, "seconds")

completed_df.unpersist()

First action time: 5.126899480819702 seconds
Second action time: 2.9843428134918213 seconds


DataFrame[order_id: bigint, order_date: date, store_id: string, product_id: string, quantity: int, unit_price: bigint, discount_pct: int, order_status: string, payment_method: string, salesperson_id: string]

In [22]:
high_quantity_df = sales_df.filter(
    F.col("quantity") > 5
).persist(StorageLevel.MEMORY_AND_DISK)

print("Total records:", high_quantity_df.count())

high_quantity_df.show(10)

high_quantity_df.unpersist()

print("Persisted data released")

Total records: 37500
+--------+----------+--------+----------+--------+----------+------------+------------+--------------+--------------+
|order_id|order_date|store_id|product_id|quantity|unit_price|discount_pct|order_status|payment_method|salesperson_id|
+--------+----------+--------+----------+--------+----------+------------+------------+--------------+--------------+
|      15|2026-01-16|     S02|       P04|       8|     17000|          15|   Completed|           UPI|         EMP04|
|      21|2026-01-22|     S04|       P04|       6|     17000|           0|    Returned|           UPI|         EMP10|
|      39|2026-02-09|     S02|       P04|       8|     17000|          15|   Completed|           UPI|         EMP04|
|      45|2026-02-15|     S04|       P04|       6|     17000|           5|   Completed|           UPI|         EMP10|
|      63|2026-03-05|     S02|       P04|       8|     17000|           0|    Returned|           UPI|         EMP04|
|      69|2026-03-11|     S04|     

In [23]:
cached_aggregation_df = sales_df.groupBy(
    "product_id"
).agg(
    F.sum("quantity").alias("total_quantity")
).cache()

cached_aggregation_df.count()

cached_aggregation_df.explain(True)

cached_aggregation_df.show()

cached_aggregation_df.unpersist()

== Parsed Logical Plan ==
'Aggregate ['product_id], ['product_id, 'sum('quantity) AS total_quantity#1715]
+- Project [order_id#31L, order_date#32, store_id#34, product_id#33, quantity#35, unit_price#40L, discount_pct#36, order_status#37, payment_method#38, salesperson_id#39]
   +- Project [product_id#33, order_id#31L, order_date#32, store_id#34, quantity#35, discount_pct#36, order_status#37, payment_method#38, salesperson_id#39, unit_price#40L]
      +- Join LeftOuter, (product_id#33 = product_id#0)
         :- Project [order_id#31L, order_date#32, product_id#33, store_id#34, quantity#35, discount_pct#36, order_status#37, payment_method#38, CASE WHEN ((order_id#31L % cast(17 as bigint)) = cast(0 as bigint)) THEN cast(null as string) ELSE concat(EMP, lpad(cast(((order_id#31L % cast(12 as bigint)) + cast(1 as bigint)) as string), 2, 0)) END AS salesperson_id#39]
         :  +- Project [order_id#31L, order_date#32, product_id#33, store_id#34, quantity#35, discount_pct#36, order_status#37,

DataFrame[product_id: string, total_quantity: bigint]

**Part G**

In [24]:
print(
    "Existing partitions:",
    sales_df.rdd.getNumPartitions()
)

repartitioned_df = sales_df.repartition(8)

print(
    "New partitions:",
    repartitioned_df.rdd.getNumPartitions()
)

Existing partitions: 2
New partitions: 8


In [25]:
partition_df = repartitioned_df.withColumn(
    "partition_id",
    F.spark_partition_id()
)

partition_df.groupBy(
    "partition_id"
).count().orderBy(
    "partition_id"
).show()

+------------+-----+
|partition_id|count|
+------------+-----+
|           0|12500|
|           1|12500|
|           2|12500|
|           3|12500|
|           4|12500|
|           5|12500|
|           6|12500|
|           7|12500|
+------------+-----+



In [26]:
store_partition_df = sales_df.repartition(
    6, "store_id"
)

store_partition_df.withColumn(
    "partition_id",
    F.spark_partition_id()
).select(
    "store_id",
    "partition_id"
).distinct().orderBy(
    "partition_id", "store_id"
).show()

+--------+------------+
|store_id|partition_id|
+--------+------------+
|     S03|           1|
|     S02|           3|
|     S01|           4|
|     S04|           5|
+--------+------------+



In [27]:
df_12 = sales_df.repartition(12)

df_coalesce = df_12.coalesce(4)

df_repartition = df_12.repartition(4)

print("Original partitions:", df_12.rdd.getNumPartitions())
print("Coalesce partitions:", df_coalesce.rdd.getNumPartitions())
print("Repartition partitions:", df_repartition.rdd.getNumPartitions())

print("Coalesce execution plan:")
df_coalesce.explain()

print("Repartition execution plan:")
df_repartition.explain()

Original partitions: 12
Coalesce partitions: 4
Repartition partitions: 4
Coalesce execution plan:
== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=true
+- == Final Plan ==
   ResultQueryStage 4
   +- Coalesce 4
      +- ShuffleQueryStage 3
         +- Exchange RoundRobinPartitioning(12), REPARTITION_BY_NUM, [plan_id=6320]
            +- *(3) Project [order_id#31L, order_date#32, store_id#34, product_id#33, quantity#35, unit_price#40L, discount_pct#36, order_status#37, payment_method#38, salesperson_id#39]
               +- *(3) BroadcastHashJoin [product_id#33], [product_id#0], LeftOuter, BuildRight, false
                  :- AQEShuffleRead local
                  :  +- ShuffleQueryStage 0
                  :     +- Exchange hashpartitioning(product_id#33, 200), ENSURE_REQUIREMENTS, [plan_id=6220]
                  :        +- *(1) Project [id#30L AS order_id#31L, date_add(2026-01-01, cast((id#30L % 180) as int)) AS order_date#32, concat(P, lpad(cast(((id#30L % 6) + 1) as string), 2,

In [28]:
sales_df.repartition(4).write \
    .mode("overwrite") \
    .option("header", True) \
    .csv("/content/sales_csv_output")

print("Four CSV part files saved successfully!")

Four CSV part files saved successfully!


**Final Challenge**

In [30]:
completed_sales_df = sales_df.filter(
    F.col("order_status") == "Completed"
).na.fill({
    "discount_pct": 0
})


joined_df = completed_sales_df.join(
    products_df,
    "product_id",
    "inner"
).join(
    stores_df,
    "store_id",
    "inner"
)

revenue_df = joined_df.withColumn(
    "gross_revenue",
    F.col("quantity") * F.col("unit_price")
).withColumn(
    "discounted_revenue",
    F.col("gross_revenue") *
    (1 - F.col("discount_pct") / 100.0)
)


aggregated_df = revenue_df.groupBy(
    "region", "category"
).agg(
    F.sum("gross_revenue").alias("total_gross_revenue"),
    F.sum("discounted_revenue").alias("total_discounted_revenue"),
    F.count("*").alias("transaction_count")
)


rank_window = Window.partitionBy(
    "region"
).orderBy(
    F.col("total_discounted_revenue").desc()
)

final_df = aggregated_df.withColumn(
    "revenue_rank",
    F.dense_rank().over(rank_window)
)


final_df = final_df.cache()

final_df.show(truncate=False)

final_df.repartition(2).write \
    .mode("overwrite") \
    .parquet("/content/final_sales_parquet")

print("Final pipeline completed successfully!")

saved_df = spark.read.parquet(
    "/content/final_sales_parquet"
)

saved_df.show(truncate=False)

print("Final aggregated records:", saved_df.count())
final_df.unpersist()

+------+-----------+-------------------+------------------------+-----------------+------------+
|region|category   |total_gross_revenue|total_discounted_revenue|transaction_count|revenue_rank|
+------+-----------+-------------------+------------------------+-----------------+------------+
|South |Electronics|2156236000         |2.0242794E9             |19484            |1           |
|South |Office     |662524000          |6.060058E8              |6495             |2           |
|South |Accessories|113631000          |1.05018025E8            |6494             |3           |
|South |Networking |87669000           |8.2684125E7             |6494             |4           |
|West  |Electronics|2493348000         |2.2992404E9             |12985            |1           |
|West  |Accessories|68169500           |6.42985E7               |6493             |2           |
|North |Electronics|934848000          |8.551824E8              |6492             |1           |
|North |Office     |441388000 

DataFrame[region: string, category: string, total_gross_revenue: bigint, total_discounted_revenue: double, transaction_count: bigint, revenue_rank: int]